In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ast import literal_eval

In [14]:
lat= [45, 45.2, 45.4, 45.6, 45.8, 46]
long= [-120, -120.2, -120.4, -120.6, -120.8, -121]
alt= [60, 58, 56, 54, 52, 50]
errorlat= 4000 #meters 
errorlong= 4000
erroralt= 4000

In [49]:
def findtheta(lat, long, alt):
    x= np.abs(lat[-1]-lat[0])*0.0175
    y= np.abs(long[-1]-long[0])*0.0175
    z= np.abs(alt[-1]-alt[0])
    return np.arctan((np.sqrt(x**2+y**2))/z)

def degtom (lat1, long1, lat2, long2, r= 6371000): #haversine equation?
    lat1rad = np.radians(lat1)
    long1rad = np.radians(long1)
    lat2rad = np.radians(lat2)
    long2rad = np.radians(long2)
    dlat = lat2rad - lat1rad
    dlong = long2rad - long1rad

    a= np.sin(dlat/2)**2 + np.cos(lat1rad)*np.cos(lat2rad)*np.sin(dlong/2)**2
    b= 2*np.arcsin(np.sqrt(a))
    
    errorla1= -r*2*(np.cos(dlat/2)*np.sin(dlat/2)-np.sin(lat1rad)*np.cos(lat2rad)*np.sin(dlong/2)**2)/(np.sqrt(1-(np.cos(lat1rad)*np.cos(lat2rad)*np.sin(dlong/2)**2+np.sin(dlat/2)**2)**2))
    errorla2= r*2*(np.cos(dlat/2)*np.sin(dlat/2)-np.cos(lat1rad)*np.sin(lat2rad)*np.sin(dlong/2)**2)/(np.sqrt(1-(np.sin(dlat/2)**2+np.cos(lat1rad)*np.cos(lat2rad)*np.sin(dlong/2)**2)**2))
    errorlo1= -r*2*(np.cos(lat1rad)*np.cos(lat2rad)*np.cos(dlong/2)*np.sin(dlong/2))/(np.sqrt(1-(np.cos(lat1rad)*np.cos(lat2rad)*np.sin(dlong/2)**2+np.sin(dlat/2)**2)**2))
    errorlo2= r*2*(np.cos(lat1rad)*np.cos(lat2rad)*np.cos(dlong/2)*np.sin(dlong/2))/(np.sqrt(1-(np.cos(lat1rad)*np.cos(lat2rad)*np.sin(dlong/2)**2+np.sin(dlat/2)**2)**2))

    errordegtom= np.sqrt((errorla1)**2*errorlat**2+(errorla2)**2*errorlat**2+(errorlo1)**2*errorlong**2+(errorlo2)**2*errorlong**2)
    print (errorla1, errorla2, errorlo1, errorlo2)
    #print (errorlo1)
    return r*b, errordegtom

def airrho(z, H=8500): #calculates density based on rho naught, height, and scale height
                                    #8.5 km is the scale height for Earth and 1.225 kg/m^3 is the density of air at sea level
    rho0= 1.225 #kg/m^3
    return (rho0*(1/(np.e**(z/H))))
    
bestfit= [] #bestfit must be defined outside of the function so that it can be used as a variable

H=8500 #meters

rockyrho = 3500 #kg/m^3
metalrho = 7500 #kg/m^3 #constants for bolide density based on possible composition
icyrho = 750 #kg/m^3

In [50]:
degtom(lat[0], long[0], lat[1], long[1])[1]

-22219.600872365776 22219.465385131905 11080.588145711941 -11080.588145711941


140454927.84452084

In [44]:
degtom(lat[0], long[0], lat[1], long[1])[1]

11080.588145711941


140454927.84452084

In [5]:
totaldlatlong

135788.43974621687

In [9]:
totalmass= 0 #error with conversion? SIGNIFICANTLY larger
totalh= 0 
totalerror= 0

for i in range (5): 
    deltalatlong = np.abs(degtom(lat[i], long[i], lat[i+1], long[i+1])) #change in lat converted to meters 
    deltaalt = (np.abs(alt[i+1]-alt[i]))*1000 #multipled by 1000 to convert to meters
    h= np.sqrt(deltalatlong**2 + deltaalt**2)
    
    totalh= totalh+h 

    altm = (alt[i]+alt[i+1])*1000
    
    rho = airrho((1/2)*altm)
    totalmass = totalmass+(h*rho)

In [7]:
rrocky = (3/4)*(totalmass/rockyrho) #+- totalerror
rmetal = (3/4)*(totalmass/metalrho)
ricy = (3/4)*(totalmass/icyrho)

print (rrocky, rmetal, ricy) #this unit is METERS 

0.058419143110383 0.0272622667848454 0.272622667848454


In [8]:
#these calculations match my hand calculations!!